# Lesson 31 — Exercises

Unless an exercise says otherwise, solve nonconvex transcriptions **locally** with `m.solve(skip_convex_check=True)` (reading, Section 3). Each exercise ends with a check cell. Assign your answers to the named variables and the checks run; they are skipped while a variable is still `None`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm

# Dynamic-optimization tools used in this lesson.
import time, logging
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from discopt.dae import ContinuousSet, DAEBuilder, FDBuilder, Trajectory, fit_trajectories
# Local NLP solves (skip_convex_check=True) carry no dual bound by design; silence that log line.
logging.getLogger("discopt.solver").setLevel(logging.ERROR)


## Exercise 1 — Order of accuracy, and a Legendre trap (10 pts)

Transcribe the logistic equation $\dot x = x(1 - x)$, $x(0) = 0.1$ on $[0, 5]$ (exact solution $x(t) = 1/(1 + 9e^{-t})$) as a pure simulation (`m.minimize(0)`) with

* `FDBuilder(..., method="backward")`,
* `DAEBuilder` with Radau $K = 2$ and $K = 3$,
* `DAEBuilder` with Gauss–Legendre $K = 2$ (`scheme="legendre"`),

for $N \in \{4, 8, 16, 32\}$ elements. Measure the error in $x(5)$ and fit the observed order $p$ (slope of $\log|\text{error}|$ against $\log N$).

1. Report `orders`. Expected: about 1, 3, 5 and 4, i.e. $2K-1$ for Radau and $2K$ for Legendre.
2. For Legendre, first read the "final" value as `x[-1, -1]`. You should see only first-order convergence. Explain why: where is the last Legendre node? Then fix it by adding `xf = m.continuous("xf", lb=-10, ub=10)` with `m.subject_to(xf == dae.state_at("x", 5.0))`, which evaluates the collocation polynomial at $t = 5$. Solve this nonlinear simulation with `skip_convex_check=True`.

In [ ]:
# TODO


In [ ]:
orders = None   # e.g. {"FD backward": 1.0, "Radau K=2": 3.0, "Radau K=3": 5.0, "Legendre K=2": 4.0}

if orders is not None:
    for name, expected in {"FD backward": 1, "Radau K=2": 3, "Radau K=3": 5, "Legendre K=2": 4}.items():
        assert abs(orders[name] - expected) < 0.5, (name, orders[name])
    print("Exercise 1 checks passed")

## Exercise 2 — Minimum time, bang-bang control, and a speed limit (15 pts)

A cart obeys $\ddot x = u$, $|u| \le 1$, and must travel from rest at $x = 0$ to rest at $x = 1$ as fast as possible {cite:p}`Betts2010`.

1. Apply the time-scaling trick from the reading (Section 5): $\tau \in [0, 1]$, decision variable `tf`, and right-hand sides multiplied by `tf`. Use Radau $K = 3$ and $N = 20$. Expected: $t_f^* = 2$ with bang-bang control ($u = +1$ then $u = -1$, switching at $t_f/2$). Store the value in `tf_even`.
2. Repeat with $N = 9$ and store `tf_odd`. Why is $t_f > 2$ now, and what does the control do in the middle element? What does this tell you about locating switching times with a fixed grid of piecewise-constant controls?
3. Add the speed limit $|v| \le 0.8$ through the state bounds ($N = 20$) and store `tf_vmax`. Derive the exact minimum time by hand: accelerate, cruise, decelerate. Expected: $t_f = 2.05$, and the grid reports slightly more. Plot $u(t)$ and $v(t)$ on the true time axis $t = \tau\,t_f$.

In [ ]:
# TODO


In [ ]:
tf_even = None   # N = 20
tf_odd = None    # N = 9
tf_vmax = None   # N = 20, |v| <= 0.8

if tf_even is not None:
    assert abs(tf_even - 2.0) < 1e-4
    assert tf_odd > 2.005
    assert 2.05 <= tf_vmax <= 2.06
    print("Exercise 2 checks passed")

## Exercise 3 — Batch reactor: ramp limits and the optimal batch time (15 pts)

Start from the batch reactor of the reading (Section 4; $N = 20$, Radau $K = 3$; unconstrained optimum $b(1) = 0.61045$).

1. **Path constraint.** A real jacket cannot change temperature instantly. Add $|T_{i+1} - T_i| \le 5$ K between consecutive elements and store the new optimum in `b_ramp`. Expected: about $0.6101$. Which part of the profile changes, and why is the loss so small?
2. **Free batch time.** Each batch also needs $t_d = 0.5$ h of downtime (charging, cleaning). Maximize the *productivity* $b(t_f)/(t_f + t_d)$ over both the temperature profile and $t_f \in [0.05, 5]$ h, using time scaling as in Exercise 2. Store `productivity` and `tf_star`. Expected: productivity $\approx 0.668$ at $t_f^* \approx 0.16$ h, a short, hot batch. How does $T(t)$ compare with the profile for $t_f = 1$?
3. **Validate** the free-time solution by simulating the optimal $T$ profile with `solve_ivp` (`method="Radau"`, tight tolerances) on the true time axis. Store `b_sim_gap` $= |b_{\text{NLP}}(t_f^*) - b_{\text{sim}}(t_f^*)|$, expected below $10^{-6}$.

*Note:* do not pass `initial_solution=` with this ratio objective. In `discopt` 0.9.0 that combination raises a `ValueError` (shape mismatch), and the default start converges anyway.

In [ ]:
# TODO


In [ ]:
b_ramp = None
productivity = None
tf_star = None
b_sim_gap = None

if b_ramp is not None:
    assert 0.6095 < b_ramp < 0.61045
    assert abs(productivity - 0.668) < 2e-3
    assert 0.13 < tf_star < 0.20
    assert b_sim_gap < 1e-6
    print("Exercise 3 checks passed")

## Exercise 4 — Kinetic parameters from two experiments (15 pts)

For $A \xrightarrow{k_1} B \xrightarrow{k_2} C$, only $B$ is measured. Experiment 1 starts from pure $A$, $(a_0, b_0) = (1, 0)$; experiment 2 starts from a mixture, $(0.5, 1.0)$. The cell below generates the data. Do not change the seed.

1. Fit $(k_1, k_2)$ to **experiment 1 only**, then to **both experiments jointly**. Use `discopt.dae.fit_trajectories` with `Trajectory(t_data, {"b": B}, {"a": a0, "b": b0})`, shared parameter variables `k1, k2 = m.continuous(...)` with bounds $[0.01, 10]$ captured in the `rhs`, `t_span=(0, 4)`, `nfe=16`, `ncp=3`, and `m.minimize(fit.least_squares())`.
2. For each fit, compute Gauss–Newton standard errors $s^2 (J^\top J)^{-1}$, with $J$ the finite-difference *trajectory* sensitivity $\partial b(t_\ell)/\partial(k_1, k_2)$ from `solve_ivp` (reading, Section 6) and $s^2 = \text{SSR}/(n - 2)$.
3. Expected: both fits within 3 standard errors of the truth $(2.0, 0.7)$, and adding experiment 2 shrinks the standard error of $k_2$ about twofold. Explain physically why starting with $B$ present informs $k_2$ {cite:p}`Bard1998`.
4. *(Discussion)* If you also try `discopt.estimate.estimate_parameters` on this model, compare its reported standard errors with yours and explain any difference: total versus partial derivatives with respect to $\theta$ when the states are tied to $\theta$ by equality constraints.

In [ ]:
# Data (given)
k1_true, k2_true, sigma = 2.0, 0.7, 0.01

def simulate_B(k1, k2, y0, t_eval):
    sol = solve_ivp(lambda t, y: [-k1 * y[0], k1 * y[0] - k2 * y[1]], (0, t_eval[-1]), y0,
                    t_eval=t_eval, rtol=1e-10, atol=1e-12)
    return sol.y[1]

rng = np.random.default_rng(31)
t_data = np.linspace(0.25, 4.0, 16)
B1 = simulate_B(k1_true, k2_true, (1.0, 0.0), t_data) + sigma * rng.standard_normal(16)
B2 = simulate_B(k1_true, k2_true, (0.5, 1.0), t_data) + sigma * rng.standard_normal(16)

In [ ]:
# TODO


In [ ]:
k_one = None    # np.array([k1, k2]) fitted to experiment 1
se_one = None   # their standard errors
k_two = None    # joint fit to experiments 1 + 2
se_two = None

if k_one is not None:
    truth = np.array([k1_true, k2_true])
    assert np.all(np.abs(k_one - truth) < 3 * se_one)
    assert np.all(np.abs(k_two - truth) < 3 * se_two)
    assert se_one[1] / se_two[1] > 1.5
    print("Exercise 4 checks passed")

## Exercise 5 — How fine a grid? Van der Pol optimal control (10 pts)

$$\min_u \int_0^5 \big(x_1^2 + x_2^2 + u^2\big)\,dt \quad\text{s.t.}\quad
\dot x_1 = (1 - x_2^2)\,x_1 - x_2 + u,\;\; \dot x_2 = x_1,\;\; x(0) = (0, 1),\;\; -0.3 \le u \le 1.$$

Use state bounds $[-5, 5]$ and `dae.integral(...)` for the objective.

1. For $K \in \{1, 2, 3\}$ and $N \in \{5, 10, 20, 40\}$, record the NLP objective $J_{\text{NLP}}$, the **simulated** objective $J_{\text{sim}}$ (integrate the ODE with the optimal piecewise-constant $u$, adding $\dot q = x_1^2 + x_2^2 + u^2$ as a third state), `m.num_variables`, and the solve time. Store everything in `results[(K, N)] = (J_nlp, J_sim, n_vars, seconds)`.
2. Expected: $K = 1$ is badly *optimistic* ($J_{\text{NLP}} \ll J_{\text{sim}}$ at small $N$). For $K = 3$, $N = 40$: $J \approx 2.8707$ with $|J_{\text{NLP}} - J_{\text{sim}}| < 10^{-6}$.
3. Which is the smallest model (fewest variables) whose state discretization error $|J_{\text{NLP}} - J_{\text{sim}}|$ is below $10^{-4}$? Separate the two error sources of reading Section 7. Why does $J_{\text{sim}}$ keep decreasing with $N$ even when $J_{\text{NLP}} = J_{\text{sim}}$?

In [ ]:
# TODO


In [ ]:
results = None   # {(K, N): (J_nlp, J_sim, n_vars, seconds)}

if results is not None:
    J3, Js3, _, _ = results[(3, 40)]
    assert abs(J3 - 2.8707) < 1e-3 and abs(J3 - Js3) < 1e-5
    J1, Js1, _, _ = results[(1, 20)]
    assert Js1 - J1 > 0.5          # implicit Euler fools the optimizer
    print("Exercise 5 checks passed")

## Exercise 6 — Switching limits for an on/off heater (5 pts)

Use the heater model of reading Section 8 ($N = 10$, $K = 2$, `m.solve(time_limit=120)`; the binary problems are convex MIQPs and solve to `optimal`, each in under about 30 s).

1. Solve the relaxation ($0 \le u \le 1$) and store `J_relaxed`. Solve the binary problem with at most $S \in \{1, 2, 4\}$ switches and with no limit. Store `J_switch = {1: ..., 2: ..., 4: ..., None: ...}`.
2. Expected: $J_{\text{relaxed}} \approx 0.193 \le J(\infty) \approx 0.233 \le J(4) \approx 0.352 \le J(2) \approx 0.574 \le J(1) \approx 0.978$. Plot $J$ against $S$ and explain the ordering. Why does the unlimited solution chatter, and what happens to the gap $J(\infty) - J_{\text{relaxed}}$ as $N$ grows (try $N = 20$ if time permits)?

In [ ]:
# TODO


In [ ]:
J_relaxed = None
J_switch = None

if J_switch is not None:
    expected = {1: 0.978, 2: 0.574, 4: 0.352, None: 0.233}
    for S, J in expected.items():
        assert abs(J_switch[S] - J) < 2e-3, (S, J_switch[S])
    assert abs(J_relaxed - 0.193) < 2e-3
    print("Exercise 6 checks passed")